In [1]:
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
import json

In [2]:
DEBUG = False #True

In [3]:
#court_considerations = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv")
#print(court_considerations.shape)
#court_considerations

laws = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv")
print(laws.shape)
laws

(175933, 3)


,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
...,...,...,...
175928,Art. 9 Abs. 4 ZWV,4 Das Grundbuchamt versieht auf Antrag des Eig...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175929,Art. 10 Abs. 1 ZWV,1 Das ARE ist im Bereich des Zweitwohnungswese...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175930,Art. 10 Abs. 2 ZWV,2 Die Baubewilligungsbehörden eröffnen dem ARE...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175931,Art. 12 ZWV,Die nachstehenden Erlasse werden wie folgt geä...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...


In [4]:
if DEBUG:
    train = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv").head(40)
    
    train['gold_citations_expanded'] = train['gold_citations'].str.split(';')
    train_expanded = train.explode('gold_citations_expanded')
    train_expanded['gold_citations_expanded'] = train_expanded['gold_citations_expanded'].str.strip()
    train_expanded = train_expanded.reset_index(drop=True).rename(columns={'gold_citations_expanded': 'gold_citation'})
    
    train_expanded = train_expanded[['query_id', 'query', 'gold_citation']]
    
    positive_labels = (
        train_expanded.groupby('query_id')['gold_citation'].agg(list).reset_index()
        .set_index('query_id')['gold_citation'].to_dict()
    )
    
else:
    train = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv")#.head(1)
train.shape

(40, 2)

Du bist ein Experte für juristische Recherche im Schweizer Recht.

Analysiere das folgende juristische Problem und extrahiere die wichtigsten rechtlichen Elemente:

- Rechtlicher Operator (z.B. Löschung, Haftung, Anspruch, Vorrang, Kündigung)
- Rechtliches Objekt (z.B. Vormerkung, Dienstbarkeit, Mietvertrag, Hilfsmittel)
- Rechtskontext / Rechtsgebiet (z.B. Grundbuch, Unfallversicherung, Mietrecht)
- Wichtige Bedingung oder Konflikt (z.B. ohne Zustimmung, konkurrierende Ansprüche)
- Relevante Rolle oder Beteiligter (z.B. Eigentümer, Versicherer, Berechtigter)

Erstelle anschließend:
1) Eine kurze juristische Zusammenfassung des Problems (1–2 Sätze).
2) Eine prägnante juristische Suchanfrage für Retrieval und Embeddings.

Regeln:
- Sprache: Deutsch
- Verwende juristische Schlüsselbegriffe
- Entferne Namen, unnötige Fakten und narrative Details
- Maximal 25 Wörter für die Suchanfrage
- Keine vollständigen Fragen

Text:
'A Swiss woman (Sofia) married, on 12 September 2011 in Lucerne, a foreign national (Marco) who had been employed as a live‑in caregiver by a relative of hers and was accommodated and paid by that household; the parties signed a prenuptial agreement providing for separation of property and a waiver of inheritance rights, they never formed a common household nor pooled finances, and the husband later produced medical certificates alleging incapacity for work and brought claims against his employers before the labour authorities. In proceedings for protective measures for the marital union, the husband seeks a maintenance payment and a provisio ad litem on the basis of alleged lack of means, while the wife alleges the marriage was a sham concluded solely to secure a residence permit, asks for a declaration of abuse of rights and the dismissal of any maintenance obligation. Should the court conclude that the husband’s maintenance claim is barred as an abuse of rights because the marriage was contracted and preserved only to circumvent residence rules, there was no community of life nor reciprocal financial contributions, and, subsidiarily, would maintenance nevertheless be payable when assessed under the criteria of Art. 176 ZGB and the rules on protective measures for the marital union?'

Antwortformat:

Rechtlicher Operator: ...
Rechtliches Objekt: ...
Rechtskontext / Rechtsgebiet: ...
Wichtige Bedingung oder Konflikt: ...
Relevante Rolle oder Beteiligter: ...

Zusammenfassung: ...
Suchanfrage: ...

In [5]:
train.values[39]

array(['test_040',
       'A Swiss woman (Sofia) married, on 12 September 2011 in Lucerne, a foreign national (Marco) who had been employed as a live‑in caregiver by a relative of hers and was accommodated and paid by that household; the parties signed a prenuptial agreement providing for separation of property and a waiver of inheritance rights, they never formed a common household nor pooled finances, and the husband later produced medical certificates alleging incapacity for work and brought claims against his employers before the labour authorities. In proceedings for protective measures for the marital union, the husband seeks a maintenance payment and a provisio ad litem on the basis of alleged lack of means, while the wife alleges the marriage was a sham concluded solely to secure a residence permit, asks for a declaration of abuse of rights and the dismissal of any maintenance obligation. Should the court conclude that the husband’s maintenance claim is barred as an abuse of ri

In [6]:
query_refinement = {
    'test_001': """**Rechtlicher Operator**:
vorsorgliche Massnahmen / Unterlassungsanordnung / Beschlagnahme / Zuständigkeit

**Rechtliches Objekt**:
Quellcode und vertrauliche Dokumentation von Softwareprogrammen; Urheberrechte und Geschäftsgeheimnisse

**Rechtskontext / Rechtsgebiet**:
Urheberrecht, Geschäftsgeheimnisschutz, Lauterkeitsrecht, Zivilprozessrecht (vorsorgliche Massnahmen), internationale Zuständigkeit

**Wichtige Bedingung oder Konflikt**:
Glaubhaftmachung eines Rechts, drohende oder bestehende Verletzung, Dringlichkeit bzw. schwer wiedergutzumachender Schaden, Verhältnismässigkeit der Massnahmen

**Relevante Rolle oder Beteiligter**:
Softwareunternehmen als Rechteinhaber (Gesuchsteller), ehemaliger Mitarbeiter, Schweizer Unternehmen als mutmasslicher Nutzer der Software, kantonale Behörden

**Zusammenfassung**:
Mehrere US-Softwareunternehmen verlangen von einer kantonalen Behörde in der Schweiz vorsorgliche Massnahmen gegen ein Schweizer Unternehmen und einen ehemaligen Mitarbeiter wegen angeblicher Nutzung und Weitergabe von Quellcode und vertraulicher Dokumentation. Zu prüfen sind die Zuständigkeit der Behörden sowie die Voraussetzungen für vorsorgliche Massnahmen wie Glaubhaftmachung des Anspruchs, drohende Rechtsverletzung, Dringlichkeit und Verhältnismässigkeit.

**Suchanfrage**:
vorsorgliche Massnahmen Schweiz Zuständigkeit kantonale Behörden Urheberrecht Geschäftsgeheimnisse Beschlagnahme Unterlassung Voraussetzungen Glaubhaftmachung Dringlichkeit Verhältnismässigkeit""",
    'test_002': """**Rechtlicher Operator:**
Verjährung von Schadenersatzansprüchen; Haftung und mögliche Exoneration wegen grober Fahrlässigkeit

**Rechtliches Objekt:**
Schadenersatzansprüche aus Verkehrsunfall (Personenschaden, Erwerbsausfall, zukünftiger wirtschaftlicher Schaden)

**Rechtskontext / Rechtsgebiet:**
Strassenverkehrsrecht und Haftpflichtrecht (SVG), insbesondere Art. 83 SVG und Art. 59 Abs. 1 SVG

**Wichtige Bedingung oder Konflikt:**
Beginn und Ablauf der Verjährungsfrist bei Kenntnis von Schaden und Schadensausmass; mögliche grobe Fahrlässigkeit des Geschädigten (Fahren gegen die Verkehrsrichtung) als Haftungsausschluss

**Relevante Rolle oder Beteiligter:**
Velofahrer als Geschädigter/Kläger; Fahrzeuglenker und Fahrzeughalter; Haftpflichtversicherer

**Zusammenfassung:**
Nach einem Verkehrsunfall macht ein Velofahrer Schadenersatzansprüche gegen Fahrzeuglenker und Versicherer geltend und erweitert diese später um weitere Schadenspositionen. Streitig sind die Verjährung der zusätzlichen Ansprüche nach Art. 83 SVG sowie eine mögliche Haftungsbefreiung wegen grober Fahrlässigkeit des Geschädigten nach Art. 59 Abs. 1 SVG.

**Suchanfrage:**
Verjährung Schadenersatz Verkehrsunfall Art. 83 SVG Kenntnis Schaden Haftungsausschluss grobe Fahrlässigkeit Velofahrer Art. 59 Abs. 1 SVG""",
    'test_003':"""**Rechtlicher Operator:**
Schuldenerlass / Forderungsreduktion; Anspruch auf Ersatz von Instandstellungs- bzw. Abnutzungskosten

**Rechtliches Objekt:**
Fahrzeugmietvertrag bzw. Leasingvertrag über ein Nutzfahrzeug; Forderung aus Vertragsverletzung und übermässiger Abnutzung

**Rechtskontext / Rechtsgebiet:**
Vertragsrecht und Mietrecht (Obligationenrecht), insbesondere Art. 267a OR; Abgrenzung zum Konsumkreditgesetz (KKG)

**Wichtige Bedingung oder Konflikt:**
Wirksamkeit eines konventionellen Schuldenerlasses ohne Annahme; Pflicht zur sofortigen Prüfung und Anzeige von Mängeln bei Rückgabe der Mietsache; Anwendbarkeit des Konsumkreditgesetzes bei geschäftlicher Nutzung

**Relevante Rolle oder Beteiligter:**
Leasing- bzw. Vermietungsunternehmen als Gläubiger; Unternehmen als Mieter; persönlich haftender Geschäftsführer/Schuldner

**Zusammenfassung:**
Nach Rückgabe eines geschäftlich genutzten Mietfahrzeugs verlangt der Vermieter Ersatz für übermässige Abnutzung und behauptet eine teilweise Forderungsreduktion. Streitpunkte sind die Wirksamkeit eines Schuldenerlasses ohne Annahme sowie der Anspruch auf Ersatzkosten trotz fehlender sofortiger Prüfung und Anzeige nach Art. 267a OR und die Anwendbarkeit des Konsumkreditgesetzes.

**Suchanfrage:**
Schuldenerlass ohne Annahme Forderungsreduktion Leasingvertrag Geschäftsfahrzeug Ersatz übermässige Abnutzung Art. 267a OR Prüfung Anzeige Rückgabe Mietobjekt Anwendbarkeit Konsumkreditgesetz""",
    'test_004': """**Rechtlicher Operator:**
Konkurseröffnung; Fristberechnung / Fristablauf; Aufhebung des Rechtsvorschlags

**Rechtliches Objekt:**
Konkursbegehren nach Betreibung; Zahlungsbefehl und Rechtsvorschlag; Entscheid über Rechtsöffnung

**Rechtskontext / Rechtsgebiet:**
Schuldbetreibungs- und Konkursrecht (SchKG), insbesondere Art. 166 Abs. 2 SchKG

**Wichtige Bedingung oder Konflikt:**
Berechnung der peremptorischen 15-Monatsfrist und deren Unterbrechung während des Rechtsöffnungsverfahrens; Zeitpunkt des Fristendes (Dispositiv vs. begründeter Entscheid); Nachweis der Zahlungsfähigkeit

**Relevante Rolle oder Beteiligter:**
Gläubiger als Gesuchsteller; Schuldnergesellschaft; Betreibungs- bzw. Konkursgericht

**Zusammenfassung:**
Nach Aufhebung eines Rechtsvorschlags stellt ein Gläubiger ein Konkursbegehren gegen eine Gesellschaft. Streitig sind die Einhaltung der peremptorischen Frist nach Art. 166 Abs. 2 SchKG unter Berücksichtigung der Fristunterbrechung sowie die Möglichkeit, den Konkurs durch Glaubhaftmachung der Zahlungsfähigkeit anzufechten.

**Suchanfrage:**
Konkursbegehren Frist Art. 166 Abs. 2 SchKG Unterbrechung Rechtsöffnungsverfahren Beginn Ende Frist Dispositiv Entscheid Zahlungsfähigkeit Schuldner Konkursaufhebung""",
    'test_005': """**Rechtlicher Operator:**
Durchsetzung eines Grundpfandrechts / Betreibung auf Pfandverwertung; Einreden gegen Forderung und Pfandhaftung

**Rechtliches Objekt:**
Inhaber-Schuldbrief als Sicherungsmittel für Kreditforderung

**Rechtskontext / Rechtsgebiet:**
Sachenrecht und Schuldbetreibungsrecht (Grundpfandrecht, Schuldbrief; Betreibung auf Pfandverwertung)

**Wichtige Bedingung oder Konflikt:**
Schuldbrief ohne Bezeichnung eines Schuldners; Frage der Novation der Grundforderung oder fiduziarische Sicherungsübertragung; Umfang der durchsetzbaren Forderung und Gültigkeit der Kündigung/Fälligkeit

**Relevante Rolle oder Beteiligter:**
Gläubiger und Inhaber des Schuldbriefs; Grundstückseigentümer bzw. Kreditnehmer; mögliche persönliche Schuldner

**Zusammenfassung:**
Ein Gläubiger hält einen Inhaber-Schuldbrief, der als Sicherheit für eine Kreditbeziehung übertragen wurde, und betreibt die Pfandverwertung des Grundstücks. Streitpunkte sind die persönliche Haftung der Beteiligten, die Wirkung des Schuldbriefs (Novation oder Sicherungsübertragung) sowie Umfang und Fälligkeit der Forderung.

**Suchanfrage:**
Inhaber-Schuldbrief Pfandverwertung persönliche Haftung Schuldnerbezeichnung Novation oder Sicherungsübertragung Einreden gegen Grundforderung Umfang Pfandhaftung Fälligkeit Kündigung Grundpfandrecht""",
    'test_006': """**Rechtlicher Operator:**
Produktehaftung / Haftung des Herstellers; Beweis eines Produktfehlers; Haftungsausschluss / Exkulpation

**Rechtliches Objekt:**
Fahrzeugkomponente (Brems- bzw. Reibbelag) als mutmasslich fehlerhaftes Produkt

**Rechtskontext / Rechtsgebiet:**
Produktehaftungsrecht nach PrHG (Bundesgesetz über die Produktehaftpflicht)

**Wichtige Bedingung oder Konflikt:**
Beweislast für Produktfehler gemäss Art. 4 PrHG; mögliche Exkulpation des Herstellers nach Art. 5 PrHG (insbesondere lit. b und e); Verjährungseinwand

**Relevante Rolle oder Beteiligter:**
Geschädigter Fahrzeughalter; Hersteller bzw. Produzent des Bauteils; Nachfolgegesellschaft des Herstellers

**Zusammenfassung:**
Ein Fahrzeughalter verlangt Schadenersatz wegen angeblich fehlerhafter Fahrzeugkomponente mit vorzeitigem Verschleiss. Streitpunkte sind der Nachweis eines Produktfehlers nach Art. 4 PrHG, die Verjährung sowie mögliche Haftungsbefreiung des Herstellers nach Art. 5 PrHG.

**Suchanfrage:**
Produktehaftung Fahrzeugkomponente Beweis Produktfehler Art. 4 PrHG Haftung Hersteller Exkulpation Art. 5 lit. b e PrHG Verjährung Schadenersatz""",
    'test_007': """**Rechtlicher Operator:**
Vertragliche Haftung aus Behandlungsvertrag / Sorgfaltspflichtverletzung; Anspruch auf Honorar bzw. Rückerstattung

**Rechtliches Objekt:**
Ärztlicher Behandlungsvertrag (Augenoperation / medizinische Behandlung)

**Rechtskontext / Rechtsgebiet:**
Vertragsrecht im Obligationenrecht, insbesondere Auftragsrecht (Arztvertrag als Auftrag)

**Wichtige Bedingung oder Konflikt:**
Qualifikation des Vertrags als Auftrag statt Werkvertrag; Verletzung der ärztlichen Sorgfaltspflicht durch fehlende Voruntersuchungen und unvollständige Behandlung; Anspruch auf Honorar trotz unvollständiger Leistung

**Relevante Rolle oder Beteiligter:**
Patientin als Auftraggeberin; behandelnder Arzt als Auftragnehmer; zweiter Facharzt als nachbehandelnder Leistungserbringer

**Zusammenfassung:**
Eine Patientin verlangt Rückerstattung von Behandlungskosten und Ersatz für Nachbehandlungen wegen angeblicher Verletzung der ärztlichen Sorgfaltspflicht bei einer Augenoperation. Streitpunkte sind die Qualifikation des Behandlungsvertrags als Auftrag, eine mögliche Pflichtverletzung des Arztes sowie dessen Anspruch auf Vergütung für teilweise erbrachte Leistungen.

**Suchanfrage:**
Arztvertrag Auftrag Sorgfaltspflichtverletzung medizinische Behandlung Haftung Arzt Kosten Nachbehandlung Vergütungsanspruch unvollständige Leistung Rückerstattung Honorar""",
    'test_008': """**Rechtlicher Operator:**
Internationale Zuständigkeit / Kompetenz zur Anordnung vorsorglicher Unterhaltsmassnahmen

**Rechtliches Objekt:**
Unterhaltsvorschuss bzw. Unterhaltsmassnahmen für minderjährige Kinder

**Rechtskontext / Rechtsgebiet:**
Internationales Privatrecht und internationales Zivilverfahrensrecht (Lugano-Übereinkommen, internationale Kindesentführung, elterliche Sorge und Kindesunterhalt)

**Wichtige Bedingung oder Konflikt:**
Mögliche widerrechtliche Verbringung der Kinder; gewöhnlicher Aufenthalt der Kinder; Konkurrenz der Zuständigkeit zwischen schweizerischen und deutschen Gerichten

**Relevante Rolle oder Beteiligter:**
Mutter als Gesuchstellerin; Vater als sorgeberechtigter Elternteil; minderjährige Kinder; schweizerische und deutsche Gerichte

**Zusammenfassung:**
Nach der Verbringung zweier Kinder aus Deutschland in die Schweiz beantragt die Mutter vorsorgliche Unterhaltsleistungen bei einem schweizerischen Gericht. Streitpunkt ist die internationale Zuständigkeit der Schweizer Behörden gegenüber der bereits angerufenen deutschen Gerichtsbarkeit.

**Suchanfrage:**
internationale Zuständigkeit Kindesunterhalt vorsorgliche Massnahmen Lugano-Übereinkommen widerrechtliche Kindesentführung gewöhnlicher Aufenthalt Kinder Konkurrenz Zuständigkeit Schweiz Deutschland""",
    'test_009': """**Rechtlicher Operator:**
Anerkennung und Vollstreckbarerklärung ausländischer Entscheidungen; Herausgabe von Bankinformationen

**Rechtliches Objekt:**
Ausländischer Nachlassentscheid und Ernennung einer Nachlassverwalterin (letters of administration); Bankkontoinformationen des Erblassers

**Rechtskontext / Rechtsgebiet:**
Internationales Privatrecht und Anerkennung ausländischer Entscheidungen; Erbrecht und Bankrecht (Bankgeheimnis)

**Wichtige Bedingung oder Konflikt:**
Möglicher Verstoss gegen den schweizerischen Ordre public wegen möglicher Bigamie; fehlende Apostille auf ausländischen Urkunden; Voraussetzungen der Anerkennung ausländischer Nachlassentscheide

**Relevante Rolle oder Beteiligter:**
Nachlassverwalterin als Gesuchstellerin; Bank als informationspflichtige Institution; erste Ehefrau und Kinder als mögliche Erben

**Zusammenfassung:**
Eine ausländische Nachlassverwalterin verlangt von einer Schweizer Bank Auskunft über Konten eines verstorbenen Kunden gestützt auf einen ausländischen Nachlassentscheid. Streitpunkte sind die Anerkennung dieser Entscheidung in der Schweiz trotz möglichem Verstoss gegen den Ordre public sowie die Bedeutung einer fehlenden Apostille.

**Suchanfrage:**
Anerkennung ausländischer Nachlassentscheidung Schweiz Ordre public Bigamie Apostille Bankauskunft Erben Nachlassverwalter internationales Privatrecht Bankgeheimnis""",
    'test_010': """**Rechtlicher Operator:**
Strafbarkeit / Qualifikation der Tat als qualifizierter Raub; Mittäterschaft; Überprüfung der Beweiswürdigung im Rechtsmittelverfahren

**Rechtliches Objekt:**
Raubdelikt unter Verwendung von Gewalt und Waffen (qualifizierter Raub)

**Rechtskontext / Rechtsgebiet:**
Strafrecht und Strafprozessrecht, insbesondere Mittäterschaft bei Raub sowie Beweiswürdigung und Überprüfung durch die Rechtsmittelinstanz (Art. 398 StPO)

**Wichtige Bedingung oder Konflikt:**
Abgrenzung zwischen Mittäterschaft am qualifizierten Raub und Teilnahme an Diebstahl; Kenntnis von Gewaltmitteln; Neubewertung von Beweisen unter Beachtung des Grundsatzes in dubio pro reo

**Relevante Rolle oder Beteiligter:**
Beschuldigter als mutmasslicher Mittäter; Mittäter/Komplizen; Opfer (Angestellter); Strafverfolgungsbehörden und Rechtsmittelinstanz

**Zusammenfassung:**
Nach einem bewaffneten Überfall auf ein Gasthaus wird ein Beteiligter als Organisator identifiziert, obwohl er behauptet, lediglich Fahrer gewesen zu sein. Streitpunkte sind die Qualifikation als Mittäter eines qualifizierten Raubes sowie die Überprüfung der Beweiswürdigung im Berufungsverfahren.

**Suchanfrage:**
qualifizierter Raub Mittäterschaft Gewaltanwendung Waffen Beweiswürdigung Berufung Art. 398 StPO in dubio pro reo Abgrenzung Diebstahl Teilnahme Strafrecht""",
    'test_011': """**Rechtlicher Operator:**
Gerichtszuständigkeit / internationale Zuständigkeit; Durchsetzung von Forderungen aus Abtretung und Handeln ohne Vollmacht

**Rechtliches Objekt:**
Forderungen aus Softwarevertriebsvertrag; Kostenersatz und Rückerstattungsansprüche aus Vertragsdurchführung

**Rechtskontext / Rechtsgebiet:**
Internationales Zivilverfahrensrecht und internationales Privatrecht (LDIP, Gerichtsstandsrecht); Vertragsrecht (Handeln ohne Vollmacht nach Art. 38–39 OR)

**Wichtige Bedingung oder Konflikt:**
Wirksamkeit und Reichweite einer Gerichtsstandsvereinbarung zugunsten eines ausländischen Forums; Wirkung der Forderungsabtretung; Haftung bei Vertretung ohne Vollmacht

**Relevante Rolle oder Beteiligter:**
Dienstleistungsunternehmen als Zessionar und Kläger; ursprünglicher Vertragspartner als Zedent; angeblicher Vertreter ohne Vollmacht; beteiligte Gesellschaften

**Zusammenfassung:**
Ein Treuhand- und Beratungsunternehmen macht in der Schweiz Forderungen aus Kosten und aus abgetretenen Ansprüchen eines Softwarevertriebsvertrags geltend. Streitpunkt ist die Zuständigkeit schweizerischer Gerichte trotz ausländischer Gerichtsstandsklausel sowie mögliche Haftung wegen Handelns ohne Vollmacht.

**Suchanfrage:**
internationale Zuständigkeit Schweiz Gerichtsstandsklausel Delaware LDIP Abtretung Forderung Vertretung ohne Vollmacht Art. 38 39 OR Durchsetzung Vertragsanspruch Kostenersatz""",
    'test_012': """**Rechtlicher Operator:**
Vertragliche Anspruchsdurchsetzung; Verjährung von Forderungen; Schuldanerkennung / Novation

**Rechtliches Objekt:**
Forderung aus laufendem Konto zwischen Genossenschaft und Mitglied; Vorschüsse und Inkassoerlöse

**Rechtskontext / Rechtsgebiet:**
Vertragsrecht (Obligationenrecht), insbesondere Auftrag und Rechenschaftspflicht (Art. 400 OR), Darlehen und Kontokorrent; Abgrenzung zu Factoring

**Wichtige Bedingung oder Konflikt:**
Qualifikation des Vertragsverhältnisses (Factoring vs. gemischter Vertrag); Nachweis der Forderung und Pflicht zur Rechnungslegung; Wirkung einer Schuldanerkennung auf Verjährung und Forderungshöhe

**Relevante Rolle oder Beteiligter:**
Genossenschaft als Dienstleister und Gläubiger; selbständiger Leistungserbringer als Mitglied und Schuldner

**Zusammenfassung:**
Eine Genossenschaft verlangt von einem Mitglied Rückzahlung von Vorschüssen aus einem Inkasso- und Abrechnungssystem. Streitpunkte sind die rechtliche Qualifikation des Vertragsverhältnisses, der Nachweis der offenen Forderung unter Berücksichtigung der Rechenschaftspflicht sowie die Wirkung einer späteren Schuldanerkennung.

**Suchanfrage:**
Factoring oder gemischter Vertrag Inkasso Auftrag Darlehen Kontokorrent Rechenschaftspflicht Art. 400 OR Beweis Forderung Verjährung Schuldanerkennung Novation offene Forderung""",
    'test_013': """**Rechtlicher Operator:**
Lohnanspruch / Anwendung Mindestlohn bzw. ortsüblicher Lohn nach Gesamtarbeitsvertrag

**Rechtliches Objekt:**
Arbeitsvertrag von Temporärarbeitern / Einsatzverträge mit Stundenlohn

**Rechtskontext / Rechtsgebiet:**
Arbeitsrecht, insbesondere Gesamtarbeitsvertrag (GAV), Temporärarbeit und öffentlich-rechtliche Mindestlohnbestimmungen

**Wichtige Bedingung oder Konflikt:**
Ausschluss der Mindestlohnbestimmungen im GAV für bestimmte Branchen; Anwendung des orts- und branchenüblichen Lohns; Allgemeinverbindlicherklärung des GAV; fehlende Gewerkschaftsmitgliedschaft der Arbeitnehmer

**Relevante Rolle oder Beteiligter:**
Temporärarbeitsunternehmen als Arbeitgeber; temporär eingesetzte Arbeitnehmer; Einsatzbetrieb; Arbeitgeberverband und tripartite kantonale Behörden

**Zusammenfassung:**
Ein Temporärarbeitsunternehmen wird mit Forderungen konfrontiert, die Löhne seiner eingesetzten Arbeitnehmer an einen höheren ortsüblichen oder kollektivvertraglichen Mindestlohn anzupassen. Streitpunkt ist die Anwendbarkeit des Gesamtarbeitsvertrags sowie der ortsüblichen Löhne trotz vertraglich vereinbarter Stundenlöhne.

**Suchanfrage:**
Temporärarbeit Mindestlohn Gesamtarbeitsvertrag Allgemeinverbindlicherklärung ortsüblicher Lohn kantonale tripartite Kommission Lohnanspruch Temporärarbeiter Anwendung GAV Branchenausnahme""",
    'test_014': """**Rechtlicher Operator:**
Leistungsanspruch aus Unfallversicherung / Qualifikation als Unfall oder unfallähnliche Körperschädigung

**Rechtliches Objekt:**
Schulterverletzung (Rotatorenmanschetten-Teilruptur / Schulterbeschwerden)

**Rechtskontext / Rechtsgebiet:**
Unfallversicherungsrecht (UVG) und Sozialversicherungsrecht (ATSG), insbesondere Art. 4 ATSG und Art. 6 UVG

**Wichtige Bedingung oder Konflikt:**
Vorliegen eines plötzlichen, ungewöhnlichen äusseren Ereignisses; widersprüchliche Angaben zum Unfallhergang; Abgrenzung zwischen degenerativer Erkrankung und unfallähnlicher Körperschädigung

**Relevante Rolle oder Beteiligter:**
Versicherter Arbeitnehmer; Unfallversicherer; behandelnde Ärzte bzw. medizinische Gutachter

**Zusammenfassung:**
Ein Versicherter verlangt Leistungen der Unfallversicherung wegen Schulterbeschwerden nach einer Sportaktivität. Streitpunkt ist, ob ein Unfall oder eine unfallähnliche Körperschädigung vorliegt oder ob die Beschwerden auf eine degenerative Erkrankung zurückzuführen sind.

**Suchanfrage:**
Unfallbegriff Art. 4 ATSG Schulterverletzung Sport Rotatorenmanschette degenerative Tendinopathie unfallähnliche Körperschädigung Art. 6 Abs. 2 UVG Leistungsanspruch Unfallversicherung""",
    'test_015': """**Rechtlicher Operator:**
Rückerstattungsanspruch / Rechenschaftspflicht; nachehelicher Unterhaltsanspruch

**Rechtliches Objekt:**
Vermögenswerte aus übergebenem Geldbetrag; ehebezogene Vermögensverwaltung

**Rechtskontext / Rechtsgebiet:**
Eherecht und Vertragsrecht (Auftragsrecht bzw. einfache Gesellschaft) im Obligationenrecht; Scheidungsrecht und nachehelicher Unterhalt

**Wichtige Bedingung oder Konflikt:**
Qualifikation des Rechtsverhältnisses (fiduziares Mandat vs. einfache Gesellschaft); Pflicht zur Rechenschaft und Verwendung der übergebenen Gelder; Anspruch auf nachehelichen Unterhalt nach Aufgabe der Erwerbstätigkeit

**Relevante Rolle oder Beteiligter:**
Ehefrau als Anspruchstellerin; Ehemann als Empfänger und Verwalter der Gelder

**Zusammenfassung:**
Nach der Scheidung verlangt eine Ehefrau die Rückerstattung eines während der Ehe übergebenen Geldbetrags, den ihr Ehemann angeblich treuhänderisch verwalten sollte. Streitpunkte sind die rechtliche Qualifikation des Vermögensverhältnisses, eine mögliche Rechenschaftspflicht sowie der Anspruch auf nachehelichen Unterhalt.

**Suchanfrage:**
Rückerstattung übergebenes Geld Ehe Mandat oder einfache Gesellschaft Rechenschaftspflicht Vermögensverwaltung Ehegatten nachehelicher Unterhalt Scheidung Anspruch Ehefrau""",
    'test_016': """Rechtlicher Operator:
Unterhaltsanspruch / Festsetzung vorsorglicher Ehegattenunterhaltsbeiträge

Rechtliches Objekt:
Ehegattenunterhalt während Trennung; Einkommen eines selbständig Erwerbenden

Rechtskontext / Rechtsgebiet:
Familienrecht (Eherecht), insbesondere vorsorgliche Massnahmen und Unterhaltsberechnung bei selbständiger Erwerbstätigkeit

Wichtige Bedingung oder Konflikt:
Ermittlung des massgeblichen Einkommens bei Selbständigerwerbenden (Durchschnittsgewinn mehrerer Jahre vs. letzter Jahresgewinn); Abzug unvermeidbarer Ausgaben; Anrechnung hypothetischen Einkommens des unterhaltsberechtigten Ehegatten

Relevante Rolle oder Beteiligter:
Selbständig erwerbender Ehegatte als Unterhaltsschuldner; getrennt lebender Ehegatte als Unterhaltsberechtigter

Zusammenfassung:
Nach der Trennung streiten Ehegatten über die Höhe eines vorsorglichen Unterhaltsbeitrags, insbesondere über die Berechnung des Einkommens eines selbständig erwerbenden Bäckereibetreibers und die Anrechnung eines hypothetischen Einkommens der Ehefrau.

Suchanfrage:
Ehegattenunterhalt Selbständigerwerbender Einkommen Durchschnitt Gewinn mehrere Jahre unvermeidbare Ausgaben hypothetisches Einkommen Ehegatte vorsorgliche Massnahmen Unterhaltsberechnung""",
    'test_017': """**Rechtlicher Operator:**
Kündigung wegen Zahlungsverzug / Ausweisung; Zulässigkeit neuer Beweismittel (Nova) im Rechtsmittelverfahren

**Rechtliches Objekt:**
Geschäftsmietvertrag über Verkaufskiosk und Lagerraum

**Rechtskontext / Rechtsgebiet:**
Mietrecht und Zivilprozessrecht, insbesondere Kündigung wegen Zahlungsrückstand und Ausweisung im summarischen Verfahren (cas clairs)

**Wichtige Bedingung oder Konflikt:**
Wirksamkeit der eingeschriebenen Mahnung und Kündigungsandrohung trotz nicht abgeholter Sendung; Voraussetzungen für Ausweisung im cas-clair-Verfahren; Zulässigkeit neuer Beweismittel im Berufungsverfahren

**Relevante Rolle oder Beteiligter:**
Vermieter als Eigentümer der Geschäftsräume; Mieter als Betreiber des Kiosks

**Zusammenfassung:**
Ein Vermieter kündigt ein Geschäftsmietverhältnis wegen Mietzinsrückständen und verlangt die Ausweisung im summarischen Verfahren. Streitpunkte sind die Wirksamkeit der Kündigung trotz nicht abgeholter Mahnschreiben sowie die Zulässigkeit neuer Beweismittel im Berufungsverfahren.

**Suchanfrage:**
Kündigung Geschäftsmiete Zahlungsverzug Mahnung eingeschriebener Brief nicht abgeholt Ausweisung cas clair Verfahren Zulässigkeit nova Berufung Mietzinsrückstand""",
    'test_018': """**Rechtlicher Operator:**
Rechtshilfe / Herausgabe von Beweismitteln; Anfechtung einer Beweisverfügung; Wahrung des rechtlichen Gehörs

**Rechtliches Objekt:**
Anordnung zur Herausgabe vertraulicher Geschäftsunterlagen (Quellcode, technische Dokumentation, Lizenzverträge)

**Rechtskontext / Rechtsgebiet:**
Internationale Rechtshilfe in Zivilsachen und Zivilprozessrecht (Beweisabnahme, Rechtsschutz gegen Verfügungen)

**Wichtige Bedingung oder Konflikt:**
Fehlender Hinweis auf Rechtsmittel und Fristen; mögliche Zulässigkeit einer verspäteten Beschwerde aus Vertrauensschutz; Recht zur Verweigerung der Herausgabe wegen Geschäfts- oder Berufsgeheimnis

**Relevante Rolle oder Beteiligter:**
Nicht am ausländischen Verfahren beteiligtes Unternehmen als Adressat der Herausgabeanordnung; kantonale Behörden bzw. Rechtshilfeinstanz; ausländisches Gericht als ersuchende Behörde

**Zusammenfassung:**
Ein Schweizer Unternehmen wird im Rahmen internationaler Rechtshilfe zur Herausgabe vertraulicher technischer Unterlagen verpflichtet. Streitig sind die Anfechtbarkeit der Verfügung, die Zulässigkeit einer verspäteten Beschwerde mangels Rechtsmittelbelehrung sowie die Wahrung des rechtlichen Gehörs hinsichtlich eines möglichen Geheimhaltungsrechts.

**Suchanfrage:**
internationale Rechtshilfe Beweisabnahme Herausgabe Geschäftsgeheimnis nicht Partei Anfechtbarkeit Verfügung fehlende Rechtsmittelbelehrung verspätete Beschwerde rechtliches Gehör Verweigerungsrecht""",
    'test_019': """**Rechtlicher Operator:**
Internationale Zuständigkeit / Anerkennung ausländischer Scheidung; Anspruch auf vorsorglichen Ehegattenunterhalt

**Rechtliches Objekt:**
Ausländischer Scheidungsentscheid und vorsorgliche Unterhaltsmassnahmen zwischen Ehegatten

**Rechtskontext / Rechtsgebiet:**
Internationales Privatrecht und Familienrecht (Anerkennung ausländischer Scheidungen; vorsorgliche Massnahmen im Scheidungsverfahren)

**Wichtige Bedingung oder Konflikt:**
Parallel laufendes Scheidungsverfahren im Ausland; Anerkennung eines ausländischen Scheidungsentscheids ohne persönliche Anwesenheit der Parteien; mögliche Verletzung des schweizerischen Ordre public

**Relevante Rolle oder Beteiligter:**
Ehefrau als Antragstellerin für Unterhalt; Ehemann als Scheidungskläger; schweizerische Gerichte; ausländische Scheidungsbehörden

**Zusammenfassung:**
Während ein Scheidungsverfahren im Ausland läuft, beantragt eine Ehefrau in der Schweiz vorsorglichen Ehegattenunterhalt. Zu klären sind die Zuständigkeit schweizerischer Gerichte für vorsorgliche Massnahmen sowie die Anerkennung eines ausländischen Scheidungsentscheids.

**Suchanfrage:**
Anerkennung ausländische Scheidung Schweiz internationales Privatrecht vorsorglicher Ehegattenunterhalt Zuständigkeit Schweizer Gericht Ordre public Scheidung ohne persönliche Anhörung""",
    'test_020': """**Rechtlicher Operator:**
Eintragung / definitive Eintragung eines Bauhandwerkerpfandrechts; Sicherung einer Werklohnforderung

**Rechtliches Objekt:**
Bauhandwerkerpfandrecht zur Sicherung von Werklohnforderungen aus Bauvertrag

**Rechtskontext / Rechtsgebiet:**
Sachenrecht und Werkvertragsrecht (Bauhandwerkerpfandrecht, Werklohnforderungen)

**Wichtige Bedingung oder Konflikt:**
Streit über Höhe der Werklohnforderung wegen Verzögerung, Mängeln und Zusatzarbeiten; Frage der Sicherung trotz bestrittenem Anspruch; Bedarf einer gerichtlichen Expertise zur Klärung der Abrechnung

**Relevante Rolle oder Beteiligter:**
Bauunternehmer als Werkunternehmer und Pfandgläubiger; Grundstückseigentümer als Besteller; Projektmanager; Subunternehmer

**Zusammenfassung:**
Ein Bauunternehmer verlangt die definitive Eintragung eines Bauhandwerkerpfandrechts zur Sicherung ausstehender Werklohnforderungen. Der Grundstückseigentümer bestreitet die Forderung wegen Verzögerungen, Mängeln und Zusatzarbeiten und verlangt eine gerichtliche Expertise zur Klärung der Abrechnung.

**Suchanfrage:**
Bauhandwerkerpfandrecht definitive Eintragung Werklohnforderung bestritten Zusatzarbeiten Bauvertrag Verzögerung Mängel Expertise Abrechnung Sicherung Anspruch""",
    'test_021': """**Rechtlicher Operator:**
Haftung / Schadenersatzanspruch aus Vertragsverletzung

**Rechtliches Objekt:**
Verlust einer Warensendung während internationalem Transport (Luxusuhrengehäuse)

**Rechtskontext / Rechtsgebiet:**
Vertragsrecht und Transportrecht, insbesondere Auftragsrecht und Haftung bei Substitution (Art. 398 Abs. 3 OR, Art. 399 Abs. 2 OR)

**Wichtige Bedingung oder Konflikt:**
Qualifikation des Vertragsverhältnisses (Spediteur/Transporteur vs. Auftrag mit Submandat); Haftung für Handlungen eines Subunternehmers; Einhaltung von Transportinstruktionen

**Relevante Rolle oder Beteiligter:**
Importeur als Auftraggeber; Spediteur/Frachtorganisator als Beauftragter; Subspediteur als Hilfsperson

**Zusammenfassung:**
Ein Importeur verlangt Schadenersatz vom Spediteur nach Verlust einer Warensendung während eines internationalen Transports. Streitpunkt ist, ob der Spediteur für den Subspediteur haftet oder nur für dessen sorgfältige Auswahl und Instruktion.

**Suchanfrage:**
Haftung Spediteur Subspediteur Auftrag Art. 398 Abs. 3 OR Art. 399 Abs. 2 OR Verlust Warensendung internationale Spedition Verantwortung Hilfsperson""",
    'test_022': """**Rechtlicher Operator:**
Internationale Zuständigkeit / Anordnung vorsorglicher Massnahmen im Familienrecht

**Rechtliches Objekt:**
Kindesschutzmassnahmen (Obhut, Besuchsrecht, Schutzmassnahmen) und vorsorgliche Massnahmen im Eheverfahren

**Rechtskontext / Rechtsgebiet:**
Internationales Privatrecht und Familienrecht (IPRG; Haager Kinderschutzübereinkommen 1996)

**Wichtige Bedingung oder Konflikt:**
Parallel laufendes Scheidungsverfahren im Ausland; gewöhnlicher Aufenthalt der Kinder in der Schweiz zum Zeitpunkt der Klageeinreichung; Anwendung von Art. 10 oder Art. 46 IPRG und Grundsatz der perpetuatio fori

**Relevante Rolle oder Beteiligter:**
Eltern als Parteien des Familienkonflikts; minderjährige Kinder; schweizerische Gerichte; ausländisches Scheidungsgericht

**Zusammenfassung:**
Nach der Einleitung eines Scheidungsverfahrens im Ausland beantragt ein Elternteil in der Schweiz vorsorgliche Massnahmen zum Schutz der Ehe und der Kinder. Zu klären ist die internationale Zuständigkeit der schweizerischen Gerichte und die Bedeutung des gewöhnlichen Aufenthalts der Kinder.

**Suchanfrage:**
internationale Zuständigkeit Kindesschutz vorsorgliche Massnahmen IPRG Art. 10 Art. 46 gewöhnlicher Aufenthalt Kinder perpetuatio fori Haager Kinderschutzübereinkommen Scheidung Ausland""",
    'test_023': """**Rechtlicher Operator:**
Haftung eines Organmitglieds / Schadenersatzpflicht nach Sozialversicherungsrecht

**Rechtliches Objekt:**
Nicht bezahlte AHV-Beiträge eines Unternehmens

**Rechtskontext / Rechtsgebiet:**
Sozialversicherungsrecht (AHVG), insbesondere Organhaftung nach Art. 52 AHVG

**Wichtige Bedingung oder Konflikt:**
Haftung eines ehemaligen Verwaltungsratsmitglieds trotz Rücktritt; gültige Zustellung von Beitragsverfügungen; Umfang solidarischer Organhaftung und differenzierte Verantwortlichkeit; Wahrung des rechtlichen Gehörs

**Relevante Rolle oder Beteiligter:**
Ehemaliges Verwaltungsratsmitglied als möglicher Haftungsschuldner; Ausgleichskasse als Gläubiger; Gesellschaft als Beitragsschuldner

**Zusammenfassung:**
Eine Ausgleichskasse verlangt von ehemaligen Verwaltungsratsmitgliedern Schadenersatz wegen nicht bezahlter AHV-Beiträge eines Unternehmens. Umstritten sind die Haftung eines bereits zurückgetretenen Organmitglieds, die Wirksamkeit der Zustellung der Beitragsverfügungen und der Umfang der Organverantwortlichkeit.

**Suchanfrage:**
Organhaftung Verwaltungsrat AHV Beiträge Art. 52 AHVG Haftung ehemaliges Organ Zustellung Beitragsverfügung solidarische Verantwortung Ausgleichskasse Schadenersatz""",
    'test_024': """**Rechtlicher Operator:**
Beweisführung / Beweisstandard im Scheidungsverfahren; richterliche Beweisabnahme; Zulässigkeit neuer Anträge im Rechtsmittelverfahren

**Rechtliches Objekt:**
Einkommen und Erwerbsfähigkeit eines selbständig erwerbenden Ehegatten; nachehelicher Unterhalt und güterrechtliche Auseinandersetzung

**Rechtskontext / Rechtsgebiet:**
Familienrecht und Zivilprozessrecht (Scheidungsverfahren, Beweisrecht)

**Wichtige Bedingung oder Konflikt:**
Anwendung des vollen zivilrechtlichen Beweisstandards statt blossen Plausibilitätsprüfung aus vorsorglichen Massnahmen; Pflicht zur Anordnung von Beweismassnahmen; Zulässigkeit neuer Begehren im Rechtsmittelverfahren

**Relevante Rolle oder Beteiligter:**
Ehegatten als Parteien im Scheidungsverfahren; erstinstanzliches Gericht und Rechtsmittelinstanz

**Zusammenfassung:**
Im Scheidungsverfahren streiten die Ehegatten über das tatsächliche Einkommen und die Erwerbsfähigkeit eines selbständig erwerbenden Ehegatten. Zu klären sind der anzuwendende Beweisstandard, die Pflicht zur Durchführung von Beweismassnahmen sowie die Zulässigkeit neuer Anträge im Rechtsmittelverfahren.

**Suchanfrage:**
Scheidungsverfahren Beweisstandard Einkommen Selbständigerwerbender Erwerbsfähigkeit medizinische Einschränkung Beweisabnahme Zeugeneinvernahme Familienrecht Rechtsmittel neue Anträge güterrechtliche Auseinandersetzung""",
    'test_025': """**Rechtlicher Operator:**
Güterrechtliche Auseinandersetzung / Aufteilung ehelicher Vermögenswerte

**Rechtliches Objekt:**
Ausländische Liegenschaft im Miteigentum; Versicherungsentschädigung; während der Ehe erworbener Personenwagen

**Rechtskontext / Rechtsgebiet:**
Familienrecht (Güterrecht der Errungenschaftsbeteiligung) und internationales Privatrecht

**Wichtige Bedingung oder Konflikt:**
Zuständigkeit eines schweizerischen Scheidungsgerichts bezüglich ausländischer Grundstücke; Verhältnis zwischen Aufhebung von Miteigentum und güterrechtlicher Liquidation; Beweisregeln und Vermutungen zur Qualifikation von Vermögenswerten als Errungenschaft

**Relevante Rolle oder Beteiligter:**
Ehegatten als Parteien des Scheidungsverfahrens und Miteigentümer der Liegenschaft

**Zusammenfassung:**
Im Scheidungsverfahren streiten die Ehegatten über die güterrechtliche Aufteilung verschiedener Vermögenswerte, darunter eine im Ausland gelegene Liegenschaft sowie eine Versicherungsleistung. Zu klären sind die Zuständigkeit des schweizerischen Gerichts für ausländische Immobilien und die Beweisregeln zur Zuordnung von Vermögenswerten zur Errungenschaft.

**Suchanfrage:**
güterrechtliche Auseinandersetzung Scheidung Errungenschaftsbeteiligung ausländische Liegenschaft Spanien Miteigentum Aufhebung Teilung Versicherungsentschädigung Vermutung Errungenschaft Beweislast Vermögenswerte""",
    'test_026': """**Rechtlicher Operator:**
Anspruch auf Herausgabe von Erträgen / nachehelicher Unterhalt / Festsetzung und Anpassung von Kindesunterhalt

**Rechtliches Objekt:**
Mietzinserträge aus gemeinsamer Liegenschaft; Ehegattenunterhalt; Kindesunterhalt und ausserordentliche Kinderkosten

**Rechtskontext / Rechtsgebiet:**
Familienrecht und Güterrecht (Errungenschaftsbeteiligung), Unterhaltsrecht für Ehegatten und Kinder

**Wichtige Bedingung oder Konflikt:**
Verteilung von Erträgen aus gemeinsamer Liegenschaft nach Trennung; Voraussetzungen für nachehelichen Unterhalt; Berechnung des Kindesunterhalts und Verpflichtung zu ausserordentlichen Kosten (Therapie, Zahnbehandlung)

**Relevante Rolle oder Beteiligter:**
Getrennt lebende Ehegatten; minderjährige Kinder; unterhaltspflichtiger Elternteil

**Zusammenfassung:**
Nach der Trennung streiten Ehegatten über die Verteilung von Mieterträgen aus einer gemeinsam gehaltenen Liegenschaft sowie über Ehegatten- und Kindesunterhalt. Zu klären sind Ansprüche auf Erträge, nachehelicher Unterhalt und Beiträge zu ausserordentlichen Kinderkosten.

**Suchanfrage:**
Errungenschaftsbeteiligung Mieterträge gemeinsame Liegenschaft Trennung Herausgabe Ertrag nachehelicher Unterhalt Voraussetzungen Kindesunterhalt Berechnung ausserordentliche Kinderkosten Therapie Zahnbehandlung""",
    'test_027': """**Rechtlicher Operator:**
Anordnung und Überprüfung von Kindesschutzmassnahmen / vorsorgliche Übertragung der Obhut

**Rechtliches Objekt:**
Elterliche Sorge und Obhut über minderjährige Kinder

**Rechtskontext / Rechtsgebiet:**
Familienrecht und Kindesschutzrecht (KESB-Massnahmen, Kindeswohl, Anhörung von Kindern)

**Wichtige Bedingung oder Konflikt:**
Gefährdung des Kindeswohls; Gewicht eines fachlichen Abklärungsberichts; mögliche Belastung der Kinder durch erneute Anhörung; Verdacht auf Misshandlung oder sexuellen Missbrauch

**Relevante Rolle oder Beteiligter:**
Kinder als betroffene Minderjährige; Mutter und Vater als Eltern; Kindesschutzbehörde bzw. Gericht; Fachstellen (Familienabklärung, medizinische Fachpersonen)

**Zusammenfassung:**
Nach Hinweisen auf Kindeswohlgefährdung und möglichen Missbrauch wird die Obhut vorsorglich dem Vater übertragen und der Kontakt zur Mutter eingeschränkt. Streitpunkt ist, ob im Überprüfungsverfahren eine erneute Anhörung der Kinder erforderlich ist oder ob die vorhandenen Fachberichte ausreichen.

**Suchanfrage:**
Kindesschutz vorsorgliche Obhutsübertragung Kindeswohl Gefährdung Anhörung Kinder erneut erforderlich Fachbericht Familienabklärung Loyalitätskonflikt Überprüfung Massnahmen Kindesinteresse""",
    'test_028': """**Rechtlicher Operator:**
Werkeigentümerhaftung / Schadenersatzanspruch

**Rechtliches Objekt:**
Eingangstür eines Gebäudes mit Glasverglasung (bauliches Werk)

**Rechtskontext / Rechtsgebiet:**
Haftpflichtrecht, insbesondere Werkeigentümerhaftung nach Art. 58 OR

**Wichtige Bedingung oder Konflikt:**
Vorliegen eines Werkmangels (ungeeignete Verglasung oder mangelhafter Unterhalt); Mitverschulden der Geschädigten; Einfluss des Alters der baulichen Anlage

**Relevante Rolle oder Beteiligter:**
Gebäudeeigentümer bzw. Verwaltungsgesellschaft als Werkeigentümer; verletzte Besucherin als Geschädigte

**Zusammenfassung:**
Eine Besucherin verletzt sich an einer Glasfüllung einer Gebäudeeingangstür. Zu klären ist, ob ein Werkmangel oder mangelhafter Unterhalt vorliegt und ob der Eigentümer nach Art. 58 OR haftet, trotz möglicher Mitverursachung durch die Geschädigte.

**Suchanfrage:**
Werkeigentümerhaftung Art. 58 OR Werkmangel Eingangstür Glasverglasung Sicherheitsglas mangelhafter Unterhalt Gebäudeeigentümer Haftung Mitverschulden Verletzung Besucher""",
    'test_029': """**Rechtlicher Operator:**
Anordnung und Überprüfung einer Erwachsenenschutzmassnahme / Bestellung einer Beistandschaft

**Rechtliches Objekt:**
Vertretungs- und Verwaltungsbeistandschaft für eine urteils- bzw. handlungsbeeinträchtigte Person

**Rechtskontext / Rechtsgebiet:**
Erwachsenenschutzrecht (Beistandschaft, Abklärungsverfahren, Gutachten) und Verfahrensrecht

**Wichtige Bedingung oder Konflikt:**
Zulässigkeit einer Beschwerde gegen Abklärungsentscheid und psychiatrisches Gutachten; Erforderlichkeit und Verhältnismässigkeit der Beistandschaft; Abgrenzung zwischen umfassender Vertretungsbeistandschaft und milderer Begleitbeistandschaft

**Relevante Rolle oder Beteiligter:**
Betroffene Person mit gesundheitlichen Einschränkungen; Erwachsenenschutzbehörde bzw. Gericht; medizinische Fachpersonen; bestellter Beistand

**Zusammenfassung:**
Eine schwer beeinträchtigte Person wehrt sich gegen die Einleitung eines Erwachsenenschutzverfahrens, die Anordnung eines psychiatrischen Gutachtens und die provisorische Bestellung eines Beistands. Zu prüfen sind die Zulässigkeit einer sofortigen Beschwerde sowie die Notwendigkeit und Verhältnismässigkeit der Beistandschaft.

**Suchanfrage:**
Erwachsenenschutz Beistandschaft Vertretung Verwaltung Beschwerde gegen Gutachten Anordnung Abklärung Verhältnismässigkeit Beistandschaft psychisches Gutachten Erwachsenenschutzrecht""",
    'test_030': """**Rechtlicher Operator:**
Festsetzung und Anpassung von Ehegattenunterhalt / Anrechnung hypothetischen Einkommens

**Rechtliches Objekt:**
Unterhaltsbeitrag zwischen Ehegatten während Trennung; Wohnkosten der Ehewohnung

**Rechtskontext / Rechtsgebiet:**
Familienrecht (vorsorgliche Massnahmen im Eheschutzverfahren; Unterhaltsberechnung)

**Wichtige Bedingung oder Konflikt:**
Zulässigkeit der rückwirkenden Anrechnung eines hypothetischen Einkommens; notwendige Anpassungsfrist zur Aufnahme einer Erwerbstätigkeit; Verteilung Existenzminimum und Überschuss unter Berücksichtigung Wohnsituation und Betreuungspflichten

**Relevante Rolle oder Beteiligter:**
Ehegatte als Unterhaltsschuldner; getrennt lebender Ehegatte als Unterhaltsberechtigter; Gericht im Eheschutzverfahren

**Zusammenfassung:**
Im Rahmen vorsorglicher Eheschutzmassnahmen streiten die Ehegatten über die Höhe des Unterhalts und die Anrechnung eines hypothetischen Einkommens der Ehefrau. Zu klären ist insbesondere, ob ein Einkommen rückwirkend angerechnet werden darf und wie Existenzminimum und Wohnkosten zu berücksichtigen sind.

**Suchanfrage:**
Eheschutz Unterhaltsberechnung hypothetisches Einkommen Anpassungsfrist Ehegattenunterhalt Existenzminimum Überschussmethode Wohnkosten Trennung Erwerbsfähigkeit Anrechnung Einkommen""",
    'test_031': """**Rechtlicher Operator:**
Anspruch auf nachehelichen Unterhalt / Festsetzung des Unterhaltsbeitrags

**Rechtliches Objekt:**
Ehegattenunterhalt nach Scheidung

**Rechtskontext / Rechtsgebiet:**
Familienrecht (Scheidungsrecht), insbesondere nachehelicher Unterhalt nach Art. 125 ZGB

**Wichtige Bedingung oder Konflikt:**
Voraussetzungen für nachehelichen Unterhalt nach langer Ehe und langer Trennung; eingeschränkte Erwerbsfähigkeit wegen Gesundheit; Wahl der Berechnungsmethode (Existenzminimum mit Überschussverteilung vs. Lebensstandardmethode)

**Relevante Rolle oder Beteiligter:**
Unterhaltsberechtigter Ehegatte mit eingeschränkter Erwerbsfähigkeit; unterhaltspflichtiger Ehegatte mit stabilem Einkommen

**Zusammenfassung:**
Nach einer langjährigen Ehe und langer Trennungsphase verlangt ein Ehegatte nachehelichen Unterhalt. Zu prüfen sind Anspruchsvoraussetzungen nach Art. 125 ZGB sowie die geeignete Methode zur Berechnung des monatlichen Unterhalts.

**Suchanfrage:**
nachehelicher Unterhalt Art. 125 ZGB lange Ehe lange Trennung eingeschränkte Erwerbsfähigkeit Unterhaltsberechnung Existenzminimum Überschussmethode Lebensstandard Scheidung""",
    'test_032': """**Rechtlicher Operator:**
Anordnung und Überprüfung von Untersuchungshaft / Haftgrund Fluchtgefahr

**Rechtliches Objekt:**
Untersuchungshaft im Strafverfahren

**Rechtskontext / Rechtsgebiet:**
Strafprozessrecht (StPO), insbesondere Voraussetzungen der Untersuchungshaft und Ersatzmassnahmen

**Wichtige Bedingung oder Konflikt:**
Vorliegen von Fluchtgefahr; Prüfung der Verhältnismässigkeit; Möglichkeit weniger einschneidender Ersatzmassnahmen (Passabgabe, Meldepflicht, Therapie)

**Relevante Rolle oder Beteiligter:**
Beschuldigte Person; Staatsanwaltschaft; Zwangsmassnahmengericht

**Zusammenfassung:**
Eine beschuldigte Person wird wegen umfangreichen Betrugsverdachts in Untersuchungshaft genommen. Streitig ist, ob Fluchtgefahr vorliegt und ob anstelle der Haft mildere Ersatzmassnahmen ausreichen würden.

**Suchanfrage:**
Untersuchungshaft Fluchtgefahr StPO Verhältnismässigkeit Ersatzmassnahmen Passabgabe Meldepflicht Betrugsverdacht Haftprüfung Zwangsmassnahmengericht""",
    'test_033': """**Rechtlicher Operator:**
Leistungsanspruch aus Unfallversicherung / Anerkennung als Berufskrankheit

**Rechtliches Objekt:**
Gesundheitsschädigung durch berufliche Exposition (Granulomatose mit Polyangiitis, Verschlechterung von Asthma)

**Rechtskontext / Rechtsgebiet:**
Unfallversicherungsrecht (UVG), insbesondere Berufskrankheit und berufliche Verschlimmerung einer Vorerkrankung

**Wichtige Bedingung oder Konflikt:**
Kausalzusammenhang zwischen beruflicher Exposition (Staub, Lösungsmittel, chemische Stoffe) und Erkrankung; Beweisgrad; Untersuchungsgrundsatz der Versicherung und Notwendigkeit eines medizinischen Gutachtens

**Relevante Rolle oder Beteiligter:**
Versicherter Arbeitnehmer / Lernender; Unfallversicherer; Arbeitgeber; medizinische Sachverständige

**Zusammenfassung:**
Ein ehemaliger Lernender verlangt Leistungen der Unfallversicherung wegen einer schweren Autoimmunerkrankung und einer Verschlimmerung seines Asthmas infolge beruflicher Exposition gegenüber Staub, Lösungsmitteln und Chemikalien. Streitig sind die Anerkennung als Berufskrankheit sowie die Frage, ob der Versicherer den Sachverhalt ausreichend abgeklärt hat.

**Suchanfrage:**
Berufskrankheit UVG berufliche Exposition Staub Lösungsmittel Autoimmunerkrankung Asthma Verschlimmerung Kausalzusammenhang Untersuchungsgrundsatz Unfallversicherung medizinisches Gutachten""",
    'test_034': """**Rechtlicher Operator:**
Eintragung / Fristberechnung für Bauhandwerkerpfandrecht

**Rechtliches Objekt:**
Bauhandwerkerpfandrecht zur Sicherung einer Werklohnforderung

**Rechtskontext / Rechtsgebiet:**
Sachenrecht und Werkvertragsrecht, insbesondere Bauhandwerkerpfandrecht nach Art. 839 ZGB

**Wichtige Bedingung oder Konflikt:**
Beginn der viermonatigen Eintragungsfrist; Zeitpunkt der letzten Arbeitsleistung bzw. endgültigen Beendigung der Bauarbeiten; verspätete oder rechtzeitige Anmeldung des Pfandrechts

**Relevante Rolle oder Beteiligter:**
Werkunternehmer (Bauunternehmen) als Pfandgläubiger; Grundstückseigentümer als Pfandschuldner

**Zusammenfassung:**
Ein Bauunternehmen verlangt die Eintragung eines Bauhandwerkerpfandrechts für ausstehende Werklohnforderungen. Streitpunkt ist, ob die Anmeldung innerhalb der viermonatigen Frist nach Beendigung der Bauarbeiten erfolgt ist.

**Suchanfrage:**
Bauhandwerkerpfandrecht Art. 839 ZGB Eintragungsfrist vier Monate letzte Arbeitsleistung Beendigung Bauarbeiten Werklohnforderung Eintragung Rechtzeitigkeit""",
    'test_035': """**Rechtlicher Operator:**
Anordnung und Aufrechterhaltung vorsorglicher Massnahmen / Frist zur Einleitung der Hauptsacheklage

**Rechtliches Objekt:**
Sicherung von Nachlassvermögen (bewegliche Vermögenswerte) im Erbfall

**Rechtskontext / Rechtsgebiet:**
Zivilprozessrecht und internationales Privatrecht (Art. 263 ZPO; Art. 89 IPRG)

**Wichtige Bedingung oder Konflikt:**
Ob ein bereits im Ausland anhängiges Hauptverfahren als Hauptsacheklage gilt; Fristwahrung für vorsorgliche Massnahmen; Zusammenhang zwischen ausländischem Erbstreit und Vermögenswerten in der Schweiz

**Relevante Rolle oder Beteiligter:**
Miterben als Parteien des Erbstreits; schweizerische Gerichte als Anordnungsbehörde für Sicherungsmassnahmen

**Zusammenfassung:**
In einem internationalen Erbstreit wurden in der Schweiz vorsorgliche Sicherungsmassnahmen über Nachlasswerte angeordnet. Streitig ist, ob ein bereits im Ausland anhängiges Verfahren als Hauptsacheklage gilt und damit das Dahinfallen der Massnahmen verhindert.

**Suchanfrage:**
vorsorgliche Massnahmen Art. 263 ZPO Hauptsacheklage Ausland anhängig Erbstreit Nachlassvermögen Schweiz Art. 89 IPRG Frist Wahrung Sicherungsmassnahmen""",
    'test_036': """**Rechtlicher Operator:**
Anordnung einer Zwangsmassnahme / Erstellung eines DNA-Profils

**Rechtliches Objekt:**
DNA-Profil eines beschuldigten Jugendlichen zu Beweiszwecken

**Rechtskontext / Rechtsgebiet:**
Strafprozessrecht und Jugendstrafverfahren (StPO), insbesondere Zwangsmassnahmen und Beweiserhebung

**Wichtige Bedingung oder Konflikt:**
Voraussetzungen für DNA-Profilierung ohne vorhandene Spuren; Verhältnismässigkeit der Massnahme; rechtliches Gehör und Akteneinsicht nach Art. 101 Abs. 1 StPO

**Relevante Rolle oder Beteiligter:**
Jugendlicher Beschuldigter; Jugenduntersuchungsrichter; Strafverfolgungsbehörden; mutmassliches Opfer

**Zusammenfassung:**
Ein Jugendlicher wird beschuldigt, einen sexuellen Übergriff mit Diebstahl begangen zu haben, woraufhin ein Richter die Erstellung eines DNA-Profils anordnet. Streitig sind die gesetzliche Grundlage, die Verhältnismässigkeit der Massnahme und die Wahrung des rechtlichen Gehörs.

**Suchanfrage:**
DNA-Profil Anordnung Jugendlicher Beschuldigter Art. 255 StPO Zwangsmassnahme Verhältnismässigkeit rechtliches Gehör Akteneinsicht Beweisermittlung""",
    'test_037': """**Rechtlicher Operator:**
Unterlassungsanspruch / Übertragung eines Domainnamens wegen Markenverletzung oder unlauterem Wettbewerb

**Rechtliches Objekt:**
Domainnamen mit ähnlicher Bezeichnung (bikes-online / bikesonline)

**Rechtskontext / Rechtsgebiet:**
Markenrecht (MSchG/LPM) und Lauterkeitsrecht (UWG/LCD), insbesondere Schutz von Kennzeichen und Domainnamen im Wettbewerb

**Wichtige Bedingung oder Konflikt:**
Verwechslungsgefahr zwischen Marke und Domainnamen; Priorität der Nutzung im schweizerischen Markt; fehlende Unterscheidungskraft einzelner Zeichenbestandteile

**Relevante Rolle oder Beteiligter:**
Markeninhaber und Onlinehändler als Kläger; konkurrierendes Unternehmen als Domaininhaber und Betreiber eines Webshops

**Zusammenfassung:**
Ein Schweizer Fahrradhändler verlangt die Unterlassung und Übertragung mehrerer ähnlich lautender Domainnamen, die von einem konkurrierenden Anbieter registriert und für einen Webshop genutzt werden. Zu prüfen sind Markenverletzung, unlauterer Wettbewerb und mögliche Übertragung der Domainnamen.

**Suchanfrage:**
Domainname Markenverletzung UWG Verwechslungsgefahr Priorität Nutzung Schweiz Domainübertragung Markenrecht Onlinehandel ähnliche Domain bikes online""",
    'test_038': """**Rechtlicher Operator:**
Schadenersatzanspruch und Genugtuung wegen unerlaubter Handlung

**Rechtliches Objekt:**
Verdienstausfall, Heilungskosten, Sachschaden (Fahrrad) und immaterieller Schaden

**Rechtskontext / Rechtsgebiet:**
Haftpflichtrecht (unerlaubte Handlung) nach Obligationenrecht

**Wichtige Bedingung oder Konflikt:**
Nachweis eines widerrechtlichen Verhaltens; adäquater Kausalzusammenhang zwischen Gewalt/Belästigung und Arbeitsunfähigkeit; Beweis der Erwerbseinbusse trotz Auslandsaufenthalt

**Relevante Rolle oder Beteiligter:**
Geschädigter Arbeitnehmer; ehemalige Partnerin als mutmassliche Schädigerin

**Zusammenfassung:**
Ein Arbeitnehmer verlangt von seiner ehemaligen Partnerin Schadenersatz und Genugtuung wegen behaupteter körperlicher Gewalt und Belästigung, die zu Arbeitsunfähigkeit und Verdienstausfall geführt haben sollen. Streitpunkte sind der Nachweis des Kausalzusammenhangs sowie die Höhe eines möglichen immateriellen Schadensersatzes.

**Suchanfrage:**
unerlaubte Handlung Schadenersatz Verdienstausfall Genugtuung Körperverletzung Belästigung Kausalzusammenhang Arbeitsunfähigkeit Beweis Haftpflichtrecht OR""",
    'test_039': """**Rechtlicher Operator:**
Feststellung und Abrechnung eines Vertragsverhältnisses / Rückerstattungs- oder Zahlungsanspruch

**Rechtliches Objekt:**
Gemeinsame Einnahmen und Kosten aus Beratungsaufträgen

**Rechtskontext / Rechtsgebiet:**
Vertragsrecht (Obligationenrecht), insbesondere einfache Gesellschaft, Auftrag oder kooperative Vertragsbeziehung

**Wichtige Bedingung oder Konflikt:**
Rechtliche Qualifikation der Zusammenarbeit; Verteilung von Einnahmen und Kosten bei getrennten Buchhaltungen; Beweislast für Forderungen sowie für Irrtum oder Betrug bei Vereinbarungen

**Relevante Rolle oder Beteiligter:**
Zwei selbständige Berater als ehemalige Geschäftspartner

**Zusammenfassung:**
Nach der Beendigung einer informellen beruflichen Zusammenarbeit streiten zwei Berater über die Abrechnung gemeinsamer Einnahmen und Kosten. Zu klären sind die rechtliche Qualifikation ihrer Kooperation, die korrekte Aufteilung der Einnahmen sowie die Beweislast für Forderungen und behauptete Irrtümer oder Täuschung.

**Suchanfrage:**
einfache Gesellschaft oder Auftrag Zusammenarbeit ohne Vertrag Einnahmen Kostenaufteilung Beweislast Forderung Irrtum Betrug Abrechnung gemeinsame Projekte""",
    'test_040': """**Rechtlicher Operator:**
Unterhaltsanspruch im Eheschutzverfahren / Einwand des Rechtsmissbrauchs

**Rechtliches Objekt:**
Ehegattenunterhalt während bestehender Ehe

**Rechtskontext / Rechtsgebiet:**
Familienrecht (Eheschutzmassnahmen nach Art. 176 ZGB) und Grundsatz des Rechtsmissbrauchs

**Wichtige Bedingung oder Konflikt:**
Behauptete Scheinehe zur Umgehung von Aufenthaltsrecht; fehlende eheliche Lebensgemeinschaft und wirtschaftliche Gemeinschaft; Voraussetzungen für Unterhaltsanspruch im Eheschutzverfahren

**Relevante Rolle oder Beteiligter:**
Ehemann als Unterhaltskläger; Ehefrau als unterhaltspflichtige Partei

**Zusammenfassung:**
Ein Ehegatte verlangt im Eheschutzverfahren Unterhalt und Prozesskostenvorschuss, während die andere Partei geltend macht, die Ehe sei lediglich zur Erlangung eines Aufenthaltsrechts geschlossen worden. Zu prüfen sind Rechtsmissbrauch und die Voraussetzungen eines Unterhaltsanspruchs nach Art. 176 ZGB.

**Suchanfrage:**
Eheschutz Unterhaltsanspruch Art. 176 ZGB Scheinehe Rechtsmissbrauch fehlende Lebensgemeinschaft Ehegattenunterhalt Aufenthaltsrecht Motivation""",
}

In [7]:
# Translate queries from English to German
import torch
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-de"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text, max_chars=500):
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for s in sentences:
        if len(current) + len(s) < max_chars:
            current += " " + s
        else:
            chunks.append(current.strip())
            current = s

    if current:
        chunks.append(current.strip())

    return chunks


def translate(text, batch_size=8):

    text = clean_text(text)
    chunks = chunk_text(text)

    translations = []

    for i in range(0, len(chunks), batch_size):

        batch = chunks[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        outputs = model.generate(
            **inputs,
            num_beams=5,
            length_penalty=1.0,
            max_new_tokens=256,
            early_stopping=True
        )

        translations.extend(
            tokenizer.batch_decode(outputs, skip_special_tokens=True)
        )

    return " ".join(translations)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

In [8]:
if True:
    german_translations = {}

    for i, row in enumerate(train.itertuples()):
        
        qid = row.query_id
        query = row.query
    
        german_translations[qid] = translate(query)

if False:
    with open("german_translations.json", "w", encoding="utf-8") as f:
        json.dump(german_translations, f, ensure_ascii=False, indent=2)

if False:
    # Load german
    with open("/kaggle/input/datasets/carlosperez97/kaggle-legal/german_translations.json", "r", encoding="utf-8") as f:
        german_translations = json.load(f)

In [9]:
import gc

def power_clean_gpu():
    # 1. Delete the heavy objects
    global model, tokenizer, inputs, generated_tokens
    
    if 'model' in globals(): del model
    if 'tokenizer' in globals(): del tokenizer
    if 'inputs' in globals(): del inputs
    if 'generated_tokens' in globals(): del generated_tokens
    
    # 2. Clear Python's internal garbage collector
    gc.collect()
    
    # 3. Clear the CUDA cache (The most important step)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() # Clears inter-process memory
        
    print("GPU Memory Cleared. You are ready to load the Retrieval Model.")

# Execute the purge
power_clean_gpu()

GPU Memory Cleared. You are ready to load the Retrieval Model.


In [10]:
train['query'] = train['query_id'].map(german_translations)
train['query'].values[0]

'Vier US-amerikanische Softwareunternehmen (NorthWave Inc., Orion Systems LLC, ClearPeak Corp. und HarborSoft Ltd.) behaupten, dass R. Silva, der bis April 2019 in der Mailänder Entwicklungseinheit von Orion Systems arbeitete und anschließend von der Schweizer Firma Lumen (CH) Sàrl eingestellt wurde, den Quellcode und die vertrauliche Dokumentation für drei Programme (Codenamen A1, B3 und B4) kopierte und dass Lumen (CH) Sàrl diese Materialien zu nutzen beabsichtigt. Die Unternehmen bitten die kantonalen Behörden in Lausanne um eine einstweilige Verfügung, die die Verwendung, Reproduktion oder Offenlegung der Programme sowie die sofortige Beschlagnahme und forensische Inspektion von Computern und Speichermedien in Lumen (CH) Sàrl und in R verbietet. Ist eine kantonale Schweizer Behörde für die Gewährung solcher einstweiliger Maßnahmen zuständig, haben die Antragsteller ihren Status als Rechteinhaber oder sonstige Rechteinhaber plausibel gemacht, um diese Ansprüche geltend zu machen, un

In [11]:
#train.sample(10)

In [12]:
train_expanded = train[['query_id', 'query']]
print(train_expanded.shape)
display(train_expanded.head())

(40, 2)


,query_id,query
0,test_001,Vier US-amerikanische Softwareunternehmen (Nor...
1,test_002,Am 9. August 2011 wurde ein 62-jähriger Radfah...
2,test_003,Am 12. März 2012 schlossen Meridian Leasing Lt...
3,test_004,"Die Orion Manufacturing AG, eine börsennotiert..."
4,test_005,"Ein Logistikunternehmen (R Ltd.), das über ein..."


In [13]:
# Queries
# {'query_id': 'query'}
queries = train_expanded[['query_id', 'query']].set_index('query_id').to_dict()['query']
print(len(queries)) # 1139

40


In [14]:
# Content
# {'citation': 'content'}

# Optimize the content for the 512-token limit
def format_legal_content(citation, title, text):
    # Clean up redundant whitespace
    citation = " ".join(citation.split()) 
    title = " ".join(title.split())
    text = " ".join(text.split())
    
    # Prepend title to text with a clear separator
    return f"### {citation}\n**Title:** {title}\n**Text:** {text}"

#laws['content'] = laws['title'] + laws['text'] 
laws['content'] = laws.apply(lambda x: format_legal_content(x['citation'], x['title'], x['text']), axis=1)

citations = laws[['citation', 'content']].set_index('citation').to_dict()['content']
print(len(citations)) # 175933

175933


In [15]:
from sklearn.model_selection import train_test_split
import random
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses

In [16]:
import os
import shutil
import json

src = "/kaggle/input/datasets/carlosperez97/legal-retriever-models/minilm-legal-finetuned-final_v2_4/minilm-legal-finetuned-final_v2_4"
dst = "/kaggle/working/minilm_legal_fixed"

# Rebuild cleanly
if os.path.exists(dst):
    shutil.rmtree(dst)

os.makedirs(os.path.join(dst, "0_Transformer"), exist_ok=True)
os.makedirs(os.path.join(dst, "1_Pooling"), exist_ok=True)

# Copy transformer files from root -> 0_Transformer
for fname in [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "sentencepiece.bpe.model",
    "spiece.model",
    "vocab.txt",
    "vocab.json",
    "merges.txt",
    "model.safetensors",
    "model-001.safetensors",
    "pytorch_model.bin",
]:
    src_file = os.path.join(src, fname)
    if os.path.exists(src_file):
        shutil.copy2(src_file, os.path.join(dst, "0_Transformer", fname))

# Standardize weights filename
old_w = os.path.join(dst, "0_Transformer", "model-001.safetensors")
new_w = os.path.join(dst, "0_Transformer", "model.safetensors")
if os.path.exists(old_w) and not os.path.exists(new_w):
    shutil.copy2(old_w, new_w)

# Copy pooling folder
pool_src = os.path.join(src, "1_Pooling")
pool_dst = os.path.join(dst, "1_Pooling")
if os.path.exists(pool_src):
    for fname in os.listdir(pool_src):
        shutil.copy2(os.path.join(pool_src, fname), os.path.join(pool_dst, fname))

# Copy ST metadata
for fname in [
    "config_sentence_transformers.json",
    "sentence_bert_config.json",
    "README.md",
]:
    src_file = os.path.join(src, fname)
    if os.path.exists(src_file):
        shutil.copy2(src_file, os.path.join(dst, fname))

# Load and patch modules.json
modules_path = os.path.join(src, "modules.json")
with open(modules_path, "r") as f:
    modules = json.load(f)

# Force the first module to point to 0_Transformer
# and keep pooling at 1_Pooling
for module in modules:
    if module["type"].endswith("Transformer"):
        module["path"] = "0_Transformer"
    elif module["type"].endswith("Pooling"):
        module["path"] = "1_Pooling"

with open(os.path.join(dst, "modules.json"), "w") as f:
    json.dump(modules, f, indent=2)

print("Patched modules.json:")
print(json.dumps(modules, indent=2))

print("\nRebuilt tree:")
for root, dirs, files in os.walk(dst):
    print(root)
    for f in files:
        print("  -", f)

Patched modules.json:
[
  {
    "idx": 0,
    "name": "0",
    "path": "0_Transformer",
    "type": "sentence_transformers.models.Transformer"
  },
  {
    "idx": 1,
    "name": "1",
    "path": "1_Pooling",
    "type": "sentence_transformers.models.Pooling"
  },
  {
    "idx": 2,
    "name": "2",
    "path": "2_Normalize",
    "type": "sentence_transformers.models.Normalize"
  }
]

Rebuilt tree:
/kaggle/working/minilm_legal_fixed
  - modules.json
  - config_sentence_transformers.json
  - README.md
  - sentence_bert_config.json
/kaggle/working/minilm_legal_fixed/1_Pooling
  - config.json
/kaggle/working/minilm_legal_fixed/0_Transformer
  - tokenizer.json
  - model.safetensors
  - tokenizer_config.json
  - model-001.safetensors
  - config.json


In [17]:
import json

with open("/kaggle/working/minilm_legal_fixed/0_Transformer/config.json", "r") as f:
    cfg = json.load(f)

print(cfg.keys())
print(cfg.get("model_type"))

dict_keys(['add_cross_attention', 'architectures', 'attention_probs_dropout_prob', 'bos_token_id', 'classifier_dropout', 'dtype', 'eos_token_id', 'hidden_act', 'hidden_dropout_prob', 'hidden_size', 'initializer_range', 'intermediate_size', 'is_decoder', 'layer_norm_eps', 'max_position_embeddings', 'model_type', 'num_attention_heads', 'num_hidden_layers', 'output_past', 'pad_token_id', 'position_embedding_type', 'tie_word_embeddings', 'transformers_version', 'type_vocab_size', 'use_cache', 'vocab_size'])
xlm-roberta


In [18]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer("/kaggle/working/minilm_legal_fixed", device=device, trust_remote_code=True)
model.max_seq_length = 768

print("Loaded successfully")
print(model)

You are trying to use a model that was created with Sentence Transformers version 5.2.3, but you're currently using version 5.2.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded successfully
SentenceTransformer(
  (0): Transformer({'max_seq_length': 768, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [19]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model is on: {model.device}")

Total parameters: 567,754,752
Model is on: cuda:0


In [20]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 768, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [21]:
#import faiss
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Define the dataset class (Lazy loader)
class CitationDataset(torch.utils.data.Dataset):
    def __init__(self, citations_dict):
        self.citation_ids = list(citations_dict.keys())
        self.citations_dict = citations_dict

    def __len__(self):
        return len(self.citation_ids)

    def __getitem__(self, idx):
        cit_id = self.citation_ids[idx]
        text = self.citations_dict[cit_id]
        return text

# Function to encode the 176k laws
def get_all_content_embeddings(model, citations_dict, batch_size=1024, debug=False):
    model.eval()
    dataset = CitationDataset(citations_dict)
    # Using 8 workers and persistent_workers to maximize CPU throughput
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=8, 
                            pin_memory=True, persistent_workers=True)
    
    all_embs = []
    count = 0
    with torch.inference_mode():
        for batch_texts in tqdm(dataloader, desc="Encoding Corpus"):
            count += 1
            if debug and count >= 100:
                break
            # BGE-base-v1.5 is very fast; we keep embs on CPU to save VRAM for FAISS
            emb = model.encode(batch_texts, 
                               task="retrieval", # jina
                               convert_to_tensor=True,
                               normalize_embeddings=True,  # important for cosine / FAISS
                               show_progress_bar=False)
            all_embs.append(emb.cpu().float())
            
    return torch.cat(all_embs, dim=0)

# Function for GPU-FAISS search
def mine_hard_negatives_torch(query_embs, content_embs, top_k=100):
    """
    Faster alternative to FAISS for datasets < 200k.
    Keeps everything in PyTorch to avoid FAISS-GPU installation headaches.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Move everything to GPU once
    # query_embs: [N, D], content_embs: [M, D]
    q = query_embs.to(device)
    c = content_embs.to(device)
    
    with torch.inference_mode():
        # 2. Compute all similarities at once
        # [N, D] @ [D, M] -> [N, M]
        scores = torch.matmul(q, c.T)
        
        # 3. Get Top-K
        top_scores, top_indices = torch.topk(scores, k=top_k, dim=1)
        
    return top_scores.cpu(), top_indices.cpu()

# Helper to chunk queries
def get_query_chunks(queries_dict, chunk_size=1024):
    items = list(queries_dict.items())
    for i in range(0, len(items), chunk_size):
        yield items[i : i + chunk_size]

In [22]:
citation_ds = CitationDataset(citations)

if False:
    # --- Execution Flow ---
    content_embeddings = get_all_content_embeddings(model, 
                                                    citations, #dict(random.sample(list(citations.items()), 10)), #citations, 
                                                    batch_size=int(2048*1.5), 
                                                    debug=False)
    print(f"Content Embeddings Shape: {content_embeddings.shape}")

    torch.save(
        {
            "embeddings": content_embeddings,
            "citation_ids": citation_ds.citation_ids
        },
        "corpus_embeddings.pt"
    )

In [23]:
data = torch.load("/kaggle/input/datasets/carlosperez97/legal-retriever-models/bge-m3-corpus.pt")
content_embeddings = data["embeddings"]
citation_ids = data["citation_ids"]
del data

In [24]:
SIM_THRESHOLD = 0.0
TOP_K = 150
#RERANKER_TOP_K = 20

In [25]:
# Retrieval
preds = {}

for chunk in get_query_chunks(queries, chunk_size=64):
    q_ids = [item[0] for item in chunk]
    q_texts = [item[1] for item in chunk] 
    
    with torch.inference_mode():
        query_embs = model.encode(
            q_texts,
            task="retrieval",
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

    scores, indices = mine_hard_negatives_torch(
        query_embs,
        content_embeddings,
        top_k=TOP_K
    )

    for i, q_id in enumerate(q_ids):
        reco = set()

        for score, idx in zip(scores[i], indices[i]):
            if score >= SIM_THRESHOLD:
                citation_id = citation_ds.citation_ids[idx]
                reco.add(citation_id)

        preds[q_id] = reco


In [26]:
# Execute the purge
power_clean_gpu()

GPU Memory Cleared. You are ready to load the Retrieval Model.


In [27]:
import os
import shutil

model_dir = "/kaggle/input/datasets/carlosperez97/legal-reranker-models/bge-legal-reranker-v2-final_3-20260310T143045Z-1-002/bge-legal-reranker-v2-final_3"
new_model_dir = "/kaggle/working/bge-legal-reranker-v2-final_3_fixed"

os.makedirs(new_model_dir, exist_ok=True)

# Copy all files
for fname in os.listdir(model_dir):
    src = os.path.join(model_dir, fname)
    dst = os.path.join(new_model_dir, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

# Rename the model file if needed
old_path = os.path.join("/kaggle/input/datasets/carlosperez97/legal-reranker-models", 
                        "model-001.safetensors")
new_path = os.path.join(new_model_dir, "model.safetensors")

if os.path.exists(old_path):
    shutil.copy2(old_path, new_path)
    print('model passed')

print("Fixed files:", os.listdir(new_model_dir))

model passed
Fixed files: ['tokenizer.json', 'model.safetensors', 'tokenizer_config.json', 'README.md', 'config.json']


In [28]:
def run_reranker_inference(
    preds,
    queries,
    corpus,
    model,
    tokenizer,
    batch_size=16,
    max_length=512,
):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    all_rankings = {}

    for qid, candidate_doc_ids in tqdm(preds.items(), desc="Reranking"):

        if qid not in queries:
            continue

        query = queries[qid]

        if not candidate_doc_ids:
            all_rankings[qid] = []
            continue

        ranking = predict_reranker(
            query=query,
            candidate_doc_ids=candidate_doc_ids,
            corpus=corpus,
            model=model,
            tokenizer=tokenizer,
            batch_size=batch_size,
            max_length=max_length,
        )

        all_rankings[qid] = ranking

    return all_rankings

import torch

def predict_reranker(
    query,
    candidate_doc_ids,
    corpus,
    model,
    tokenizer,
    batch_size=16,
    max_length=512,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    # Keep only candidate ids that exist in corpus
    candidate_doc_ids = [doc_id for doc_id in candidate_doc_ids if doc_id in corpus]

    results = []

    for i in range(0, len(candidate_doc_ids), batch_size):
        batch_keys = candidate_doc_ids[i:i + batch_size]
        batch_docs = [corpus[key] for key in batch_keys]
        batch_queries = [query] * len(batch_docs)

        inputs = tokenizer(
            batch_queries,
            batch_docs,
            padding=True,
            truncation="only_second",
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            if device == "cuda":
                with torch.cuda.amp.autocast():
                    logits = model(**inputs).logits.view(-1)
            else:
                logits = model(**inputs).logits.view(-1)

            batch_scores = logits.float().cpu().tolist()

        results.extend(zip(batch_keys, batch_scores))

        del inputs, logits
        if device == "cuda":
            torch.cuda.empty_cache()

    results = sorted(results, key=lambda x: x[1], reverse=True)
    return results


from tqdm import tqdm

def run_reranker_inference(
    preds,
    queries,
    corpus,
    model,
    tokenizer,
    batch_size=16,
    max_length=512,
):
    """
    preds: dict {qid: [candidate_doc_id1, candidate_doc_id2, ...]}
    queries: dict {qid: query_text}
    corpus: dict {doc_id: doc_text}

    Returns:
        all_rankings: dict {qid: [(doc_id, score), ...]}
    """
    all_rankings = {}

    for qid, candidate_doc_ids in tqdm(preds.items(), desc="Inference"):
        if qid not in queries:
            continue

        query = queries[qid]

        if not candidate_doc_ids:
            all_rankings[qid] = []
            continue

        final_ranking = predict_reranker(
            query=query,
            candidate_doc_ids=list(candidate_doc_ids),
            corpus=corpus,
            model=model,
            tokenizer=tokenizer,
            batch_size=batch_size,
            max_length=max_length
        )

        all_rankings[qid] = final_ranking

    return all_rankings

In [29]:
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if True:
    model_name =  "/kaggle/working/bge-legal-reranker-v2-final_3_fixed"
    
    model = CrossEncoder(
        model_name,
        max_length=1024+512,
        device="cuda"
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)

You are trying to use a model that was created with Sentence Transformers version 5.2.3, but you're currently using version 5.2.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [30]:
RERANKER_TOP_K1 = 50
RERANKER_TOP_K2 = 20
DO_RERANKER1 = True

In [31]:
if DO_RERANKER1:
    all_rankings_ = run_reranker_inference(
        preds=preds,
        queries=queries,
        corpus=citations,
        model=model,
        tokenizer=tokenizer,
        batch_size=32,
        max_length=1024+512,
    )

    # Keep top  of reranking items 
    preds = {
        qid: ranking[:RERANKER_TOP_K1]
        for qid, ranking in all_rankings_.items()
    }
    #print(top_rankings)

Inference:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipykernel_23/1057515368.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Inference: 100%|██████████| 40/40 [03:34<00:00,  5.36s/it]


In [32]:
# Execute the purge
power_clean_gpu()

GPU Memory Cleared. You are ready to load the Retrieval Model.


In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F

class Qwen3Reranker:
    """Qwen3-Reranker-0.6B using the correct AutoModelForCausalLM approach."""

    SYSTEM_PROMPT = (
        'Judge whether the Document meets the requirements based on the '
        'Query and the Instruct provided. Note that the answer can only be '
        '"yes" or "no".'
    )

    def __init__(self, model_name, device='cuda', dtype=torch.float16):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            padding_side='left',
            trust_remote_code=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=True,
        ).to(device).eval()

        self.device = device
        self.yes_id = self.tokenizer.convert_tokens_to_ids('yes')
        self.no_id = self.tokenizer.convert_tokens_to_ids('no')

    def _format_pair(self, instruction, query, document):
        return (
            f'<|im_start|>system\n{self.SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n'
            f'<Instruct>: {instruction}\n'
            f'<Query>: {query}\n'
            f'<Document>: {document}\n'
            f'<|im_end|>\n'
            f'<|im_start|>assistant\n<think>\n\n</think>\n\n'
        )

    @torch.no_grad()
    def predict(self, pairs, batch_size=8, instruction=None):
        """Score query-document pairs. Returns list of relevance scores (0-1)."""
        if instruction is None:
            instruction = TASK_INSTRUCTION

        all_scores = []
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            prompts = [
                self._format_pair(instruction, q, d[:TEXT_TRUNCATE])
                for q, d in batch
            ]
            inputs = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=4096,
                return_tensors='pt',
            ).to(self.device)

            logits = self.model(**inputs).logits[:, -1, :]
            yes_no = torch.stack(
                [logits[:, self.no_id], logits[:, self.yes_id]], dim=1
            )
            probs = F.softmax(yes_no, dim=1)
            scores = probs[:, 1].cpu().numpy()  # P(yes)
            all_scores.extend(scores.tolist())

        return all_scores


def rerank_candidates(query, candidates):
    """Rerank (citation, text) candidates using Qwen3-Reranker."""
    if not candidates:
        return []
    pairs = [[query, text] for _, text in candidates]
    scores = reranker.predict(pairs, batch_size=RERANKER_BATCH_SIZE)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(cit, sc) for (cit, _), sc in scored]


#RERANKER_MODEL_NAME = 'Qwen/Qwen3-Reranker-0.6B'
#RERANKER_BATCH_SIZE = 8

RERANKER_MODEL_NAME = 'Qwen/Qwen3-Reranker-4B'
RERANKER_BATCH_SIZE = 2

DEVICE_RERANK = 'cuda:1'

print(f'Loading reranker: {RERANKER_MODEL_NAME} on {DEVICE_RERANK}')
reranker = Qwen3Reranker(RERANKER_MODEL_NAME, device=DEVICE_RERANK)
print('Reranker ready')

Loading reranker: Qwen/Qwen3-Reranker-4B on cuda:1


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Reranker ready


In [34]:
TASK_INSTRUCTION = (
    'Given a legal question, retrieve relevant Swiss law articles and court decisions.'
)
TEXT_TRUNCATE = 5500       # max chars for reranker input

all_rankings = {}
for qid, candidate_doc_ids in tqdm(preds.items(), desc="Inference"):
    if qid not in queries:
        continue

    query = query_refinement[qid] #queries[qid]

    if not candidate_doc_ids:
        all_rankings[qid] = []
    
    # 6. Rerank
    if DO_RERANKER1:
        reranked = rerank_candidates(query, [(item[0], citations[item[0]]) for item in candidate_doc_ids] )
    else:
        reranked = rerank_candidates(query, [(item, citations[item]) for item in candidate_doc_ids] )
        
    all_rankings[qid] = reranked

Inference: 100%|██████████| 40/40 [13:17<00:00, 19.94s/it]


In [35]:
# Keep top  of reranking items 
top_rankings = {
    qid: ranking[:RERANKER_TOP_K2]
    for qid, ranking in all_rankings.items()
}
#print(top_rankings)

In [36]:
import pandas as pd

sub = []

total_tp = 0
total_fp = 0
total_fn = 0

for qid, pred_list in top_rankings.items(): #top_rankings.items(): #preds.items():

    pred_list = list([item[0] for item in pred_list])
    #pred_list = list( pred_list)

    sub.append({
        "query_id": qid,
        "predicted_citations": ";".join(pred_list)
    })

    if DEBUG:
        gt = set(positive_labels[qid])
        pred = set(pred_list)

        tp = len(gt & pred)
        fp = len(pred - gt)
        fn = len(gt - pred)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        #print(f"{qid} | P={tp/len(pred) if len(pred)>0 else 0:.3f} "
        #      f"R={tp/len(gt) if len(gt)>0 else 0:.3f}")

sub = pd.DataFrame(sub)

if DEBUG:
    # ---- GLOBAL METRICS ----
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print("\n==== GLOBAL METRICS ====")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

sub = pd.DataFrame(sub)
sub.to_csv('/kaggle/working/submission.csv', index=False)

In [37]:
sub

,query_id,predicted_citations
0,test_001,Art. 10 IPRG;Art. 65 URG;Art. 109 Abs. 2 IPRG;...
1,test_002,Art. 83 Abs. 1 SVG;Art. 59 Abs. 1 SVG;Art. 1 A...
2,test_003,Art. 939 Abs. 2 ZGB;Art. 116 Abs. 1 OR;Art. 78...
3,test_004,Art. 166 Abs. 2 SchKG;Art. 188 Abs. 2 SchKG;Ar...
4,test_005,Art. 860 Abs. 2 ZGB;Art. 847 Abs. 1 ZGB;Art. 8...
5,test_006,Art. 4 Abs. 1 PrHG;Art. 5 Abs. 1 PrHG;Art. 9 P...
6,test_007,Art. 398 Abs. 2 OR;Art. 398 Abs. 1 OR;Art. 394...
7,test_008,Art. 79 Abs. 1 IPRG;Art. 10 IPRG;Art. 71 Abs. ...
8,test_009,Art. 27 Abs. 1 IPRG;Art. 17 IPRG;Art. 175 IPRG...
9,test_010,Art. 10 Abs. 3 StPO;Art. 140 Abs. 4 StGB;Art. ...


In [38]:
import torch
import gc

def clean_gpu():
    # 1. Clear out the variable references
    # Note: Replace 'model' and 'trainer' with your actual variable names
    global model, trainer, train_dataloader
    
    #if 'model' in globals(): del model
    if 'trainer' in globals(): del trainer
    #if 'train_dataloader' in globals(): del train_dataloader
    
    # 2. Trigger Python's garbage collector
    gc.collect()
    
    # 3. Clear the PyTorch CUDA cache
    torch.cuda.empty_cache()
    
    # Optional: Check VRAM to confirm
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

# Run it
clean_gpu()

GPU Memory Allocated: 2175.22 MB
GPU Memory Reserved: 2188.00 MB


In [39]:
import shutil
import os

for path in ["/kaggle/working/minilm_legal_fixed", "/kaggle/working/bge-legal-reranker-v2-final_3_fixed"]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Directory deleted")
    else:
        print("Directory does not exist")


Directory deleted
Directory deleted
